# Notebook 10-COLAB — Entrenamiento Oficial BETO con Validación Cruzada Estratificada (K=5)
### Entorno: Google Colab — Runtime T4 GPU | Con checkpoints reanudables

---

## Objetivo
Fine-tuning de BETO con **StratifiedKFold (K=5)** + `RandomUnderSampler` por fold,
registrando F1-Score Macro, Tiempo, RAM y VRAM.

## Sistema de reanudacion
Despues de completar cada fold, el notebook guarda en Drive:
- `beto_kfold_estado.json` — metricas acumuladas y tiempo por fold
- `beto_checkpoints/fold_N/` — pesos del modelo ganador de ese fold

Si la sesion de Colab se cierra o expira, **ejecuta todas las celdas de nuevo**.
El notebook detectara automaticamente cuantos folds ya estan completos y
reanudara desde el siguiente, sin repetir trabajo.

### Formato de salida esperado
```
BETO: F1-Score: 0.XX | Tiempo: XX min | RAM: X.XX GB | VRAM: X.XX GB
```

---
### Instrucciones
1. Selecciona **Runtime > Change runtime type > T4 GPU**.
2. Sube `train_val_set.csv` a: `MyDrive/ProyectoFinal_ML/data/`
3. Ejecuta las celdas en orden. Para reanudar: ejecuta todo de nuevo.

## Celda 0 — Verificacion del Hardware (GPU T4)

In [ ]:
!nvidia-smi

import torch
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("\n" + "=" * 55)
print("  VERIFICACION DE HARDWARE")
print("=" * 55)
print(f"  Dispositivo activo : {device}")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"  GPU                : {props.name}")
    print(f"  VRAM total         : {vram_gb:.1f} GB")
    torch.cuda.empty_cache()
    print("  Cache GPU          : limpiada")
else:
    raise RuntimeError(
        "GPU no detectada. Ve a Runtime > Change runtime type > T4 GPU."
    )
print("=" * 55)

## Celda 1 — Instalacion de Dependencias y Parche torchvision

In [ ]:
!pip install transformers datasets accelerate memory_profiler imbalanced-learn -q

# --- PARCHE CRITICO PARA COLAB ---
# La libreria 'datasets' de HuggingFace intenta importar torchvision.io.VideoReader
# al formatear tensores, pero esa clase no existe en la version de torchvision
# preinstalada en Colab. Desactivar esta flag previene el ImportError.
# Debe ejecutarse ANTES de crear cualquier Dataset de HuggingFace.
import datasets.config
datasets.config.TORCHVISION_AVAILABLE = False

import transformers, datasets, memory_profiler, imblearn

print(f"transformers    : {transformers.__version__}")
print(f"datasets        : {datasets.__version__}")
print(f"memory_profiler : {memory_profiler.__version__}")
print(f"imbalanced-learn: {imblearn.__version__}")
print(f"torchvision fix : datasets.config.TORCHVISION_AVAILABLE = False")
print("Dependencias listas.")

## Celda 2 — Montaje de Drive y Carga del Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import pandas as pd

# --- CONFIGURACION: ajusta esta ruta si es necesario ---
BASE_PATH        = '/content/drive/MyDrive/ProyectoFinal_ML/data'
TRAIN_PATH       = os.path.join(BASE_PATH, 'train_val_set.csv')
ESTADO_PATH      = os.path.join(BASE_PATH, 'beto_kfold_estado.json')
CHECKPOINTS_PATH = os.path.join(BASE_PATH, 'beto_checkpoints')
# -------------------------------------------------------

os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(
        f"No se encontro el archivo:\n  {TRAIN_PATH}\n"
        "Sube 'train_val_set.csv' a la carpeta indicada en Drive."
    )

df = pd.read_csv(TRAIN_PATH, sep=';')
df['texto_beto'] = df['texto_beto'].fillna('')

print("Dataset cargado.")
print(f"  Total de registros : {len(df):,}")
print(f"  Distribucion original:")
print(df['label'].value_counts().to_string())
print(f"\n  Estado de checkpoints : {ESTADO_PATH}")
print(f"  Carpeta de modelos    : {CHECKPOINTS_PATH}")

## Celda 3 — Importaciones y Configuracion Global

In [ ]:
import time
import warnings
import gc
import json
import shutil
import numpy as np

from memory_profiler import memory_usage
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from imblearn.under_sampling import RandomUnderSampler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset

warnings.filterwarnings('ignore')

# --- CONSTANTES GLOBALES ---
MODELO_BETO = 'dccuchile/bert-base-spanish-wwm-cased'
MAX_LENGTH  = 128
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 2e-5
K_FOLDS     = 5
SEED        = 42
# ---------------------------

print(f"  Modelo      : {MODELO_BETO}")
print(f"  Max tokens  : {MAX_LENGTH} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS} | LR: {LR}")
print(f"  K-Folds     : {K_FOLDS} | Seed: {SEED}")

## Celda 4 — Funciones de Estado (Guardar / Cargar progreso)

In [ ]:
def guardar_estado(f1_lista, vram_lista, tiempos_lista):
    """
    Persiste el progreso acumulado en Drive despues de cada fold.
    tiempos_lista: lista de segundos reales por fold.
    """
    estado = {
        'folds_completados' : len(f1_lista),
        'f1_por_fold'       : f1_lista,
        'vram_por_fold'     : vram_lista,
        'tiempos_por_fold_s': tiempos_lista,
    }
    with open(ESTADO_PATH, 'w', encoding='utf-8') as fh:
        json.dump(estado, fh, indent=2)
    print(f"  [CHECKPOINT] Estado guardado: {len(f1_lista)}/{K_FOLDS} folds completos.")


def cargar_estado():
    """
    Carga el estado previo desde Drive.
    Retorna (f1_lista, vram_lista, tiempos_lista, fold_inicio) o listas vacias.
    """
    if os.path.exists(ESTADO_PATH):
        with open(ESTADO_PATH, 'r', encoding='utf-8') as fh:
            estado = json.load(fh)
        n = estado['folds_completados']
        print(f"\n  [REANUDACION] Estado previo encontrado: {n}/{K_FOLDS} folds ya completos.")
        for i, f1 in enumerate(estado['f1_por_fold']):
            t_min = estado['tiempos_por_fold_s'][i] / 60
            print(f"    Fold {i+1}: F1={f1:.4f} | Tiempo={t_min:.1f} min")
        return (
            estado['f1_por_fold'],
            estado['vram_por_fold'],
            estado['tiempos_por_fold_s'],
            n
        )
    print("  [INICIO FRESCO] No se encontro estado previo. Empezando desde el Fold 1.")
    return [], [], [], 0


def resetear_estado():
    """
    Elimina el archivo de estado y los checkpoints para empezar desde cero.
    ADVERTENCIA: esto borra todo el progreso guardado.
    """
    if os.path.exists(ESTADO_PATH):
        os.remove(ESTADO_PATH)
    if os.path.exists(CHECKPOINTS_PATH):
        shutil.rmtree(CHECKPOINTS_PATH)
        os.makedirs(CHECKPOINTS_PATH, exist_ok=True)
    print("Estado y checkpoints eliminados. Listo para empezar desde cero.")


# --- Descomenta la siguiente linea SOLO si quieres empezar desde cero ---
# resetear_estado()

print("Funciones de estado definidas.")

## Celda 5 — Tokenizador y Funciones Auxiliares

In [ ]:
print(f"Descargando tokenizador: {MODELO_BETO}")
tokenizer = AutoTokenizer.from_pretrained(MODELO_BETO)
print("Tokenizador listo.")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'f1_macro': f1_score(labels, preds, average='macro')}


def construir_hf_dataset(textos, etiquetas):
    """Tokeniza y empaqueta en Dataset de HuggingFace."""
    raw = Dataset.from_dict({'texto': list(textos), 'labels': list(etiquetas)})

    def tokenize_fn(examples):
        return tokenizer(
            examples['texto'],
            padding='max_length',
            truncation=True,
            max_length=MAX_LENGTH
        )

    tokenized = raw.map(tokenize_fn, batched=True, remove_columns=['texto'])
    tokenized.set_format('torch')
    return tokenized


print("compute_metrics y construir_hf_dataset definidas.")

## Celda 6 — Bucle K-Fold con Checkpoints Reanudables

### Flujo por fold
```
Cargar estado (Drive) --> saltar folds ya completos
      |
      v (solo folds pendientes)
Split estratificado
  |-- TRAIN (80%) --> RandomUnderSampler --> TRAIN BALANCEADO --> BETO fine-tuning
  `-- VAL   (20%) --> sin modificar --> evaluacion F1-Score Macro
      |
      v
Guardar modelo fold en Drive --> Guardar estado JSON en Drive
```

> Para reanudar tras un corte: ejecuta el notebook completo desde el principio.
> Los folds completos se saltaran automaticamente.

In [ ]:
textos    = df['texto_beto'].tolist()
etiquetas = df['label'].tolist()

# El mismo SKF con el mismo seed siempre produce los mismos splits
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
todos_los_splits = list(skf.split(textos, etiquetas))

# --- Cargar progreso previo ---
f1_por_fold, vram_por_fold, tiempos_por_fold, fold_inicio = cargar_estado()

if fold_inicio >= K_FOLDS:
    print("\nTodos los folds ya estan completos. Salta a la Celda 7 para ver resultados.")
else:
    print(f"\nReanudando desde el Fold {fold_inicio + 1} de {K_FOLDS}.")


def ejecutar_folds_pendientes():
    """
    Ejecuta solo los folds que aun no estan en el estado guardado.
    Envuelta por memory_profiler para medir RAM del sistema.
    """
    for fold_idx in range(fold_inicio, K_FOLDS):
        train_idx, val_idx = todos_los_splits[fold_idx]

        print(f"\n{'='*60}")
        print(f"  FOLD {fold_idx + 1} / {K_FOLDS}")
        print(f"{'='*60}")

        t_fold_inicio = time.time()

        # --- Particion cruda ---
        textos_train_raw = [textos[i] for i in train_idx]
        labels_train_raw = [etiquetas[i] for i in train_idx]
        textos_val       = [textos[i] for i in val_idx]
        labels_val       = [etiquetas[i] for i in val_idx]

        print(f"  Train crudo: {len(textos_train_raw):,} | Val: {len(textos_val):,}")

        # --- RandomUnderSampler SOLO sobre train ---
        rus = RandomUnderSampler(random_state=SEED)
        idx_array = np.array(range(len(textos_train_raw))).reshape(-1, 1)
        idx_res, labels_train = rus.fit_resample(idx_array, labels_train_raw)
        textos_train = [textos_train_raw[i[0]] for i in idx_res]

        print(f"  Train balanceado: {len(textos_train):,}")
        print(f"  Dist.: {dict(zip(*np.unique(labels_train, return_counts=True)))}")

        # --- Tokenizacion ---
        print("  Tokenizando...")
        ds_train = construir_hf_dataset(textos_train, labels_train)
        ds_val   = construir_hf_dataset(textos_val,   labels_val)

        # --- Modelo BETO fresco ---
        print("  Cargando BETO en GPU...")
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

        model_fold = AutoModelForSequenceClassification.from_pretrained(
            MODELO_BETO,
            num_labels=2,
            ignore_mismatched_sizes=True
        )
        model_fold.to(device)

        # Los checkpoints durante el entrenamiento van a /content (rapido, temporal)
        # El modelo ganador del fold se copia a Drive al terminar (persistente)
        output_dir_local = f'/content/beto_fold_{fold_idx + 1}'

        training_args = TrainingArguments(
            output_dir=output_dir_local,
            num_train_epochs=EPOCHS,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE,
            learning_rate=LR,
            warmup_ratio=0.1,
            weight_decay=0.01,
            eval_strategy='epoch',
            save_strategy='epoch',
            load_best_model_at_end=True,
            metric_for_best_model='f1_macro',
            greater_is_better=True,
            logging_steps=50,
            fp16=True,
            dataloader_num_workers=0,
            report_to='none',
            seed=SEED
        )

        trainer = Trainer(
            model=model_fold,
            args=training_args,
            train_dataset=ds_train,
            eval_dataset=ds_val,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        print("  Entrenando...")
        trainer.train()

        # --- Metricas del fold ---
        metricas = trainer.evaluate()
        f1_fold  = metricas['eval_f1_macro']
        vram_gb  = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
        t_fold_s = time.time() - t_fold_inicio

        f1_por_fold.append(f1_fold)
        vram_por_fold.append(vram_gb)
        tiempos_por_fold.append(t_fold_s)

        print(f"  F1-Score Fold {fold_idx+1}: {f1_fold:.4f}")
        print(f"  VRAM pico         : {vram_gb:.2f} GB")
        print(f"  Tiempo fold       : {t_fold_s/60:.1f} min")

        # --- Guardar modelo del fold en Drive (persistente) ---
        fold_drive_path = os.path.join(CHECKPOINTS_PATH, f'fold_{fold_idx + 1}')
        os.makedirs(fold_drive_path, exist_ok=True)
        model_fold.save_pretrained(fold_drive_path)
        tokenizer.save_pretrained(fold_drive_path)
        print(f"  Modelo fold guardado en Drive: {fold_drive_path}")

        # --- Guardar estado JSON en Drive (permite reanudar si se corta) ---
        guardar_estado(f1_por_fold, vram_por_fold, tiempos_por_fold)

        # --- Limpieza de VRAM ---
        del model_fold, trainer, ds_train, ds_val
        if os.path.exists(output_dir_local):
            shutil.rmtree(output_dir_local)
        torch.cuda.empty_cache()
        gc.collect()


# --- EJECUCION con medicion de RAM (solo sobre los folds pendientes) ---
if fold_inicio < K_FOLDS:
    print(f"\nIniciando entrenamiento...")
    print(f"Corpus: {len(textos):,} | Epocas/fold: {EPOCHS} | Batch: {BATCH_SIZE} | lr: {LR}")

    t_sesion_inicio = time.time()

    muestras_ram = memory_usage(
        (ejecutar_folds_pendientes, [], {}),
        interval=1.0,
        include_children=True,
        max_usage=False,
        retval=False
    )

    ram_sesion_gb = max(muestras_ram) / 1024
    print(f"\nFolds completados en esta sesion.")
    print(f"RAM pico esta sesion: {ram_sesion_gb:.2f} GB")
else:
    ram_sesion_gb = 0.0

print("\nCelda 6 finalizada. Continua a la Celda 7 para ver resultados.")

## Celda 7 — Resultados Finales

> Esta celda lee directamente del archivo de estado en Drive.
> Funciona aunque no hayas ejecutado la Celda 6 en esta sesion.

In [ ]:
# Leer estado final desde Drive (funciona incluso en sesiones nuevas)
f1_final_lista, vram_final_lista, tiempos_final_lista, n_completos = cargar_estado()

if n_completos < K_FOLDS:
    print(f"\nAun faltan {K_FOLDS - n_completos} fold(s). Ejecuta la Celda 6 para continuar.")
else:
    tiempo_total_min = sum(tiempos_final_lista) / 60
    f1_final         = float(np.mean(f1_final_lista))
    vram_pico_gb     = max(vram_final_lista)
    ram_nota         = ram_sesion_gb if 'ram_sesion_gb' in dir() else float('nan')

    print("\n" + "=" * 65)
    print("  RESULTADOS FINALES — BETO K=5 Estratificado + Submuestreo")
    print("  Entorno: Google Colab — T4 GPU Runtime")
    print("=" * 65)
    if not np.isnan(ram_nota):
        print(f"  BETO: F1-Score: {f1_final:.2f} | Tiempo: {tiempo_total_min:.1f} min | RAM: {ram_nota:.2f} GB | VRAM: {vram_pico_gb:.2f} GB")
    else:
        print(f"  BETO: F1-Score: {f1_final:.2f} | Tiempo: {tiempo_total_min:.1f} min | VRAM: {vram_pico_gb:.2f} GB")
        print("  (RAM: ejecuta la Celda 6 completa en una sola sesion para registrarla)")
    print("=" * 65)

    print("\n  Detalle por fold:")
    for i in range(K_FOLDS):
        t_min = tiempos_final_lista[i] / 60
        print(f"    Fold {i+1}: F1={f1_final_lista[i]:.4f} | "
              f"Tiempo={t_min:.1f} min | VRAM={vram_final_lista[i]:.2f} GB")
    print(f"  Promedio F1 : {f1_final:.4f}")
    print(f"  Desv. std   : {float(np.std(f1_final_lista)):.4f}")

    resumen = pd.DataFrame({
        'Modelo':          ['BETO (bert-base-spanish-wwm-cased)'],
        'F1-Score Macro':  [round(f1_final, 4)],
        'Desv. Std':       [round(float(np.std(f1_final_lista)), 4)],
        'Tiempo total (min)': [round(tiempo_total_min, 1)],
        'VRAM Pico (GB)':  [round(vram_pico_gb, 2)],
        'K-Folds':         [K_FOLDS],
        'Epocas/Fold':     [EPOCHS],
        'Batch Size':      [BATCH_SIZE],
        'Max Length':      [MAX_LENGTH],
        'LR':              [LR],
        'Balanceo train':  ['RandomUnderSampler']
    })
    display(resumen.set_index('Modelo'))

## Celda 8 — Entrenamiento Final sobre Corpus Completo y Exportacion a Drive

> Ejecuta esta celda **una sola vez**, cuando todos los K=5 folds esten completos.
> Produce el modelo de produccion entrenado en el 100% del corpus balanceado.

In [ ]:
_, _, _, n_completos_check = cargar_estado()

if n_completos_check < K_FOLDS:
    raise RuntimeError(
        f"Solo hay {n_completos_check}/{K_FOLDS} folds completos.\n"
        "Completa todos los folds antes de ejecutar esta celda."
    )

print("Entrenamiento final sobre corpus completo balanceado...")

# Submuestreo sobre el corpus completo
rus_final = RandomUnderSampler(random_state=SEED)
idx_todos = np.array(range(len(textos))).reshape(-1, 1)
idx_bal, labels_bal = rus_final.fit_resample(idx_todos, etiquetas)
textos_bal = [textos[i[0]] for i in idx_bal]

print(f"  Corpus original  : {len(textos):,}")
print(f"  Corpus balanceado: {len(textos_bal):,}")
print(f"  Distribucion     : {dict(zip(*np.unique(labels_bal, return_counts=True)))}")

ds_completo = construir_hf_dataset(textos_bal, labels_bal)

torch.cuda.empty_cache()
model_final = AutoModelForSequenceClassification.from_pretrained(
    MODELO_BETO, num_labels=2, ignore_mismatched_sizes=True
)
model_final.to(device)

args_final = TrainingArguments(
    output_dir='/content/beto_final',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    save_strategy='no',
    fp16=True,
    dataloader_num_workers=0,
    report_to='none',
    seed=SEED
)

trainer_final = Trainer(
    model=model_final,
    args=args_final,
    train_dataset=ds_completo,
    compute_metrics=compute_metrics
)

trainer_final.train()
print("Entrenamiento final completado.")

# Exportar modelo + tokenizador a Drive
DRIVE_MODEL_PATH = os.path.join(BASE_PATH, 'beto_final_model')
os.makedirs(DRIVE_MODEL_PATH, exist_ok=True)
model_final.save_pretrained(DRIVE_MODEL_PATH)
tokenizer.save_pretrained(DRIVE_MODEL_PATH)
print(f"Modelo final exportado: {DRIVE_MODEL_PATH}")

# Exportar tabla de resultados
PATH_CSV = os.path.join(BASE_PATH, 'resultados_beto_kfold.csv')
resumen.to_csv(PATH_CSV, index=False, sep=';')
print(f"Resultados guardados  : {PATH_CSV}")
print("\nTodo exportado a Google Drive correctamente.")